In [1]:
import ee
import time
import geemap
import os

In [2]:
# ee.Authenticate()  # Uncomment if needed
ee.Initialize(project='e4e-mangrove')

print("Initializing Generalization Test Download Pipeline...")

Initializing Generalization Test Download Pipeline...


In [3]:
# ==========================================
# 1. Define Test Regions
# ==========================================
# Each region is a (name, bounding box [west, south, east, north]) tuple.
# These are major mangrove areas outside of Florida, chosen to test
# how well the model generalizes to unseen geographies.

TEST_REGIONS = [
    # Sumatra west coast — dense mangrove estuaries
    ('Indonesia_Sumatra',   [103.2, -1.5, 104.5, 0.5]),
    # Kalimantan (Borneo) coast
    ('Indonesia_Kalimantan',[116.0,  1.0, 117.5, 2.5]),
    # Sundarbans delta (Bangladesh / West Bengal)
    ('Sundarbans',          [ 88.8, 21.5,  90.0, 22.5]),
    # Yucatan / Campeche coast, Mexico
    ('Mexico_Campeche',     [-91.5, 18.5, -90.0, 19.5]),
]

# Max tiles to download per region (keeps the test set small)
MAX_TILES_PER_REGION = 3

print(f"Regions to download: {[r[0] for r in TEST_REGIONS]}")
print(f"Max tiles per region: {MAX_TILES_PER_REGION}")

Regions to download: ['Indonesia_Sumatra', 'Indonesia_Kalimantan', 'Sundarbans', 'Mexico_Campeche']
Max tiles per region: 3


In [4]:
# ==========================================
# 2. ESA WorldCover Label
# ==========================================
esa = ee.ImageCollection('ESA/WorldCover/v200').first()

# Full 11-class label (becomes Band 69 — same format as Florida training data)
esa_all_classes = esa.select('Map').byte().rename('ESA_MultiClass_Label')

# Strict mangrove mask (Class 95) — used only for tile filtering
esa_mangrove_filter = esa.eq(95)

In [5]:
# ==========================================
# 3. Download Loop (one region at a time)
# ==========================================
out_dir = os.path.join(os.getcwd(), 'Generalization_Test_Dataset')
os.makedirs(out_dir, exist_ok=True)
print(f"Output directory: {out_dir}\n")

for region_name, bbox in TEST_REGIONS:
    print(f"{'='*50}")
    print(f"Region: {region_name}")
    
    region_geom = ee.Geometry.Rectangle(bbox)

    # --- Build the 69-band stack for this region ---
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(region_geom)
          .filterDate('2023-01-01', '2024-01-01')
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
          .median())

    s2_optical = s2.select(['B4', 'B3', 'B2', 'B8']).rename(['Red', 'Green', 'Blue', 'NIR'])

    embeddings = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
                  .filterBounds(region_geom)
                  .filterDate('2023-01-01', '2024-01-01')
                  .mosaic())

    # Stack: Optical (1-4) + Embeddings (5-68) + Full Multi-Class Labels (69)
    stack = s2_optical.addBands(embeddings).addBands(esa_all_classes)

    # --- Filter grid to tiles that actually contain mangroves ---
    grid = region_geom.coveringGrid('EPSG:3857', 20480)
    tiles_with_stats = esa_mangrove_filter.reduceRegions(
        collection=grid,
        reducer=ee.Reducer.max(),
        scale=100,
        tileScale=4
    )
    mangrove_tiles = tiles_with_stats.filter(ee.Filter.gt('max', 0))

    def get_coords_gps(tile):
        return ee.Feature(tile).geometry().transform('EPSG:4326', 1).coordinates()

    coords_array = mangrove_tiles.toList(500).map(get_coords_gps).getInfo()
    total_tiles = len(coords_array)
    print(f"Mangrove tiles found: {total_tiles} — downloading up to {MAX_TILES_PER_REGION}")

    region_dir = os.path.join(out_dir, region_name)
    os.makedirs(region_dir, exist_ok=True)

    for i, coords in enumerate(coords_array[:MAX_TILES_PER_REGION]):
        geom = ee.Geometry.Polygon(coords)
        filename = os.path.join(region_dir, f'{region_name}_Tile_{i+1:03d}.tif')

        print(f"  [{i+1}/{min(MAX_TILES_PER_REGION, total_tiles)}] Downloading {os.path.basename(filename)}...")
        try:
            geemap.download_ee_image(
                image=stack,
                filename=filename,
                region=geom,
                crs='EPSG:3857',
                scale=10
            )
            print(f"  Saved: {filename}")
        except Exception as e:
            print(f"  Failed tile {i+1}: {e}")

print("\nAll downloads finished!")

Output directory: c:\vscode workspace\ml-mangrove\Scale Adaption\GEE\Generalization_Test_Dataset

Region: Indonesia_Sumatra
Mangrove tiles found: 57 — downloading up to 3
  [1/3] Downloading Indonesia_Sumatra_Tile_001.tif...


c:\Users\adytc\anaconda3\envs\mangrove\Lib\site-packages\geemap\common.py:12434: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


  0%|          |0/690 tiles [00:00<?]

c:\Users\adytc\anaconda3\envs\mangrove\Lib\site-packages\geedim\image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'None'.
  return STACClient().get(self.id)


  Saved: c:\vscode workspace\ml-mangrove\Scale Adaption\GEE\Generalization_Test_Dataset\Indonesia_Sumatra\Indonesia_Sumatra_Tile_001.tif
  [2/3] Downloading Indonesia_Sumatra_Tile_002.tif...


...

  Saved: c:\vscode workspace\ml-mangrove\Scale Adaption\GEE\Generalization_Test_Dataset\Mexico_Campeche\Mexico_Campeche_Tile_001.tif
  [2/3] Downloading Mexico_Campeche_Tile_002.tif...


  0%|          |0/690 tiles [00:00<?]

  Saved: c:\vscode workspace\ml-mangrove\Scale Adaption\GEE\Generalization_Test_Dataset\Mexico_Campeche\Mexico_Campeche_Tile_002.tif
  [3/3] Downloading Mexico_Campeche_Tile_003.tif...


  0%|          |0/690 tiles [00:00<?]

  Saved: c:\vscode workspace\ml-mangrove\Scale Adaption\GEE\Generalization_Test_Dataset\Mexico_Campeche\Mexico_Campeche_Tile_003.tif

All downloads finished!
